The CDC data summarizes the health measures of the census tract, but there are 285 MSSAs that are spread across multiple census tracts. Hence we merge the mssa fields to CDC data based on the GEOLOCATION field

In [0]:
cdc_places_bronze_df = spark.read.table("ca_healthcare_fac_bronze.cdc_places_ca_data.cdc_places_ca_data").toPandas()
print(cdc_places_bronze_df.columns)

In [0]:
cdc_places_bronze_df.shape

In [0]:
mssa_bronze_pddf = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
print(mssa_bronze_pddf.columns)

In [0]:
print(type(cdc_places_bronze_df['Geolocation'].iloc[0]))  # confirms it's a string
print(cdc_places_bronze_df['Geolocation'].iloc[0])

cdc places data have Point data but as a string but not as a shapely POINT object

In [0]:
from shapely import wkt

cdc_places_bronze_df['Geolocation'] = cdc_places_bronze_df['Geolocation'].apply(wkt.loads)



In [0]:
import geopandas as gpd

cdc_places_gpd = gpd.GeoDataFrame(cdc_places_bronze_df, geometry='Geolocation', crs="EPSG:4326")
cdc_places_gpd.head()


In [0]:
from shapely.geometry import Polygon

def convert_polygon(rings):
    return Polygon(rings[0])

mssa_bronze_pddf['geometry_rings'] = mssa_bronze_pddf['geometry_rings'].apply(convert_polygon)
mssa_bronze_pddf.head()


In [0]:
mssa_geo_gdf = gpd.GeoDataFrame(mssa_bronze_pddf[['TRACTCE', 'GEOID', 'MSSAID', 'MSSANM', 'geometry_rings']], geometry='geometry_rings', crs='EPSG:4326')
mssa_geo_gdf.head()


In [0]:
cdc_places_mssa = cdc_places_gpd.sjoin_nearest(mssa_geo_gdf, how='left')

In [0]:
cdc_places_mssa.shape

In [0]:
cdc_places_mssa = cdc_places_mssa.drop_duplicates(subset=['LocationID',
       'CategoryID', 'MeasureId', 'DataValueTypeID'])
cdc_places_mssa.shape

In [0]:
cdc_places_mssa['geolocation_lat'] = cdc_places_mssa['Geolocation'].apply(
    lambda g: g.y if g is not None else None
)
cdc_places_mssa['geolocation_lon'] = cdc_places_mssa['Geolocation'].apply(
    lambda g: g.x if g is not None else None
)
cdc_places_mssa.drop(['Geolocation'], axis=1, inplace=True)

In [0]:
cdc_places_spark = spark.createDataFrame(cdc_places_mssa)

In [0]:
cdc_places_spark.write.mode('overwrite').saveAsTable('ca_healthcare_fac_silver.cdc_places_silver.cdc_places_mssa')